## algorithm design and anlysis-2026 spring  homework 2
**Deadline**：2026.5.20

**name**:


note：
---
1. 本题目为在线OJ作业，OJ平台题目链接：https://www.nowcoder.com/acm/contest/129481，访问密码见课件；
3. 在OJ平台运行通过后，将源码复制到本文件对应题目的下方代码框中；
4. 如若作答有雷同，全部取消成绩；


## A 排序

In [ ]:
"""
解题思路：
本题的目标是通过三种“魔法”实现排序：

加法：所有数字同时 + x 相当于所有数字同时平移;再mod n;
异或：所有数字同时 xor x 相当于控制哪些“二进制位”要翻转  x ^ 1  → 改变最低位
交换：交换两个石板的值 交换值为a,b的石板,而不是位置;

本质问题:如何通过这三种操作,把a[i]这个数组的值变成i;

方法 
想交换 x 和 y
先用 + / xor 把 x → a 把 y → b
再 swap(a,b)
再把 a b 映射回 x y 变回来

困难在于无法随意改变两个数字之间的关系
但是可以改变a b之间的关系 找到 a 和 b 最低不同位

于是引入lowbit
l = lowbit(a - b)
所有操作（+ / xor) 不会改变 % l 的分组结构

eg a = 6 = 110           b = 2 = 010
    a - b = 4 = 100.     l = 4
    它们在“第3位”第一次不同

所以需要去凑：
(x 和 y 的结构) == (a 和 b 的结构)
从而进行升序排序

Step 1.   确定结构
l = lowbit(a-b) 决定分组

把所有数按.  x % l分组
举例l=4
组0.  0,4,8/4,0,8,...
组1.  1,5,9,...
组2.  2,6,10,...
组3.  3,7,11,...
所有操作+ / xor.  不会打乱这种分组结构
而swap(a,b) 可以在组内产生作用

Step 2.  调低位  + / xor
让 o[i] % l == i % l  把每个数放到正确“组”
如果做不到  输出 -1

Step 3. 检查每组是否合法
每组必须包含正确的数 i, i+l, i+2l...

Step 4. 组内排序
用：搬运 + / xor 把x,y映射到 a,b
swap(a,b)
实现任意交换
再映射回去
swap(x,y)=(+k) → swap(a,b) → ‘’、(+n-k)
"""
import sys
sys.setrecursionlimit(1000000)

def main():
    data = list(map(int, sys.stdin.buffer.read().split()))
    if not data:
        return

    n = data[0]
    swap_val_a = data[1]   # 魔法操作固定交换的两个值
    swap_val_b = data[2]
    perm = data[3:3 + n]   # 当前排列

    ops = []   # 操作记录

    def lowbit(x):
        return x & -x    # 取 x 的最低有效位。

    # group_size: a 和 b 二进制上第一个不同位的权值，决定"分组粒度"
    diff = (swap_val_a - swap_val_b + n) % n
    group_size = lowbit(diff) if diff != 0 else n

    # 三种基础操作
    def apply_swap_ab():
        """操作0 交换排列中值为 swap_val_a 和 swap_val_b 的两个位置"""
        ops.append(0)
        for i in range(n):
            if perm[i] == swap_val_a:
                perm[i] = swap_val_b
            elif perm[i] == swap_val_b:
                perm[i] = swap_val_a

    def apply_add(offset):
        """操作2 所有元素加 offset 模 n"""
        offset %= n
        if offset == 0:
            return
        ops.append(offset)
        for i in range(n):
            perm[i] = (perm[i] + offset) % n

    def apply_xor(mask):
        """操作1 所有元素异或 mask"""
        if mask == 0:
            return
        ops.append(-mask)   # 负数表示异或，输出时转换
        for i in range(n):
            perm[i] ^= mask

    #  calc_pair：找到与 (x,y) 结构等价的 (pa, pb) 
    #  使得可以用 apply_swap_ab 间接交换 x 和 y
    def calc_mapped_pair(src, dst):
        """
        给定要交换的两个值 src, dst
        返回与 (swap_val_a, swap_val_b) 结构等价的映射点 (mapped_src, mapped_dst)
        原名 calc_pair(x, y) → pa, pb
        """
        remaining_gap = (dst - src + n - group_size + n) % n

        mapped_src = 0 
        mapped_dst = 0 

        step = n // 2
        while step >= 2 * group_size:
            if remaining_gap >= step:
                remaining_gap -= step
                mapped_dst += step // 2
            else:
                mapped_src += step // 2
            step //= 2

        mapped_src += n // 2
        low_bits = src & (group_size - 1)   # src 的低位余数（余数组编号）
        mapped_src += low_bits
        mapped_dst += low_bits

        return mapped_src, mapped_dst

    # ── do_swap：实现任意两个值的交换 ─────────────────────────
    def swap_values(val_c, val_d):
        same_parity = (val_c // group_size) % 2 == (val_d // group_size) % 2

        if same_parity:
            # 同奇偶块，无法直接交换，借助中间值 bridge
            in_even_block = (val_c // group_size) % 2 == 0
            low = val_c & (group_size - 1)
            bridge = low + group_size if in_even_block else low   # 原名 p

            # swap(c, bridge); swap(d, bridge); swap(c, bridge) == swap(c, d)
            swap_values(val_c, bridge)
            swap_values(val_d, bridge)
            swap_values(val_c, bridge)

        else:
            # 不同奇偶块，映射后直接交换
            ref_src, ref_dst = calc_mapped_pair(swap_val_a, swap_val_b)  
            map_c, map_d    = calc_mapped_pair(val_c, val_d)             

            # 把整个排列平移+异或
            apply_add((map_c - val_c + n) % n)
            apply_xor(map_c ^ ref_src)
            apply_add((swap_val_a - ref_src + n) % n)

            apply_swap_ab()

            # 逆操作还原坐标系
            apply_add((ref_src - swap_val_a + n) % n)
            apply_xor(map_c ^ ref_src)
            apply_add((val_c - map_c + n) % n)

    # perm_calc：仅用加法/异或对齐低位余数 
    def calc_alignment_ops(residues, length):
        """
        递归计算只用加法/异或让 residues[i] % group_size == i % group_size 的操作序列
        原名 perm_calc(arr, length)
        返回 (is_possible, op_list)
        op_list 中正数=加法，负数=异或
        """
        if length == 1:
            return True, []

        half = length // 2

        # 按下标奇偶拆分，右移一位后递归（剥去当前最低位）
        even_residues = [residues[i * 2]     // 2 for i in range(half)] 
        odd_residues  = [residues[i * 2 + 1] // 2 for i in range(half)] 

        ok_even, even_ops = calc_alignment_ops(even_residues, half) 
        if not ok_even:
            return False, []

        ok_odd, odd_ops = calc_alignment_ops(odd_residues, half)  
        if not ok_odd:
            return False, []

        merged_ops = [] 

        # 当前层最低位是否需要初始对齐
        if residues[0] % 2:
            merged_ops.append(1 if length == 2 else -1)

        # 把偶数子树的操作"升一位"翻译到当前层
        even_xor_acc = 0   # 累积偶数子树的异或偏移
        for op in even_ops:
            if op > 0:
                # 子树的加法 → 当前层：先异或1再加1
                merged_ops.append(-1)
                merged_ops.append(1)
            else:
                # 子树的异或 op（负数）→ 当前层：异或 op*2
                merged_ops.append(op * 2)
                even_xor_acc ^= (-op * 2)

        if even_xor_acc:
            merged_ops.append(-even_xor_acc)

        # 把奇数子树的操作"升一位"翻译到当前层
        odd_xor_acc = 0 
        for op in odd_ops:
            if op > 0:
                # 子树的加法 → 当前层：先加1再异或1
                merged_ops.append(1)
                merged_ops.append(-1)
            else:
                merged_ops.append(op * 2)
                odd_xor_acc ^= (-op * 2)

        # 检查奇偶子树高位异或累积是否一致（一致才能合并）
        if (odd_xor_acc & half) != (even_xor_acc & half):
            return False, []

        # 去掉高位后再次检查
        even_xor_acc %= half
        odd_xor_acc  %= half
        if even_xor_acc != odd_xor_acc:
            return False, []

        # 合并相邻异或操作（两个连续异或可以合并为一次）
        compressed = []
        for op in merged_ops:
            if compressed and op < 0 and compressed[-1] < 0:
                combined = -((-compressed[-1]) ^ (-op))
                if combined == 0:
                    compressed.pop()
                else:
                    compressed[-1] = combined
            else:
                compressed.append(op)

        return True, compressed

    # 主流程 
    # Step 1：用加法/异或让每个元素落入正确的余数组
    if group_size > 1:
        initial_residues = [perm[i] & (group_size - 1) for i in range(group_size)]
        ok, alignment_ops = calc_alignment_ops(initial_residues, group_size)
        if not ok:
            print(-1)
            return
        for op in alignment_ops:
            if op > 0:
                apply_add(op)
            else:
                apply_xor(-op)

    # Step 2：验证每个余数组内的元素集合是否合法
    for group_id in range(group_size):   
        group_vals = sorted(perm[group_id::group_size])
        expected   = list(range(group_id, n, group_size))
        if group_vals != expected:
            print(-1)
            return

        # Step 3：组内排序，把每个元素换到正确位置
        for pos in range(group_id, n, group_size):   
            if perm[pos] != pos:
                swap_values(pos, perm[pos])

    # 最终校验
    if perm != list(range(n)):
        print(-1)
        return

    # 输出 
    output_lines = [str(len(ops))]
    for op in ops:
        if op == 0:
            output_lines.append("0")
        elif op < 0:
            output_lines.append(f"1 {-op}")
        else:
            output_lines.append(f"2 {op}")

    sys.stdout.write("\n".join(output_lines))


if __name__ == "__main__":
    main()

## B 长跑

In [ ]:
"""
解题思路:
标注所有的关键点: 起点0, 所有的补给点, 终点 total_dist
设 min_cost[i] 表示: 到达第 i 个关键点并在这里补充满体力, 最少需要多少金币
设 cheapest[pos] 表示: 同一个位置可能有多个补给站, 只保留花费最小的那个

让 min_cost 从前往后转移:
如果能从位置 i 直接走到位置 j(距离 <= stamina_limit)
    如果 j 是终点, 不需要花钱
    如果 j 是补给点, 再花 cheapest[j]
最后记录到达终点的最少花费 ans
如果 coins >= ans, 则可以跑完全程, 否则不行
"""

import sys
from collections import deque

INF = 10 ** 18


def main():
    data = sys.stdin.read().split()
    idx = 0
    out = []

    while idx < len(data):
        n = int(data[idx])
        idx += 1
        total_dist = int(data[idx])
        idx += 1
        stamina_limit = int(data[idx])
        idx += 1
        coins = int(data[idx])
        idx += 1

        cheapest = {}

        for _ in range(n):
            pos = int(data[idx])
            idx += 1
            cost = int(data[idx])
            idx += 1

            # 起点和终点不用作为付费补给点处理
            if 0 < pos < total_dist:
                if pos not in cheapest:
                    cheapest[pos] = cost
                else:
                    cheapest[pos] = min(cheapest[pos], cost)

        # 起点直接到终点
        if total_dist <= stamina_limit:
            out.append("Yes")
            continue

        # 关键点：起点 + 补给点 + 终点
        points = [(0, 0)]

        for pos in sorted(cheapest):
            points.append((pos, cheapest[pos]))

        points.append((total_dist, 0))

        m = len(points)
        dp = [INF] * m
        dp[0] = 0

        q = deque()
        q.append(0)

        for j in range(1, m):
            pos_j, cost_j = points[j]

            # 把距离太远的点弹出
            while q and pos_j - points[q[0]][0] > stamina_limit:
                q.popleft()

            # 当前点可以由队首这个最小 dp 转移过来
            if q:
                if j == m - 1:
                    dp[j] = dp[q[0]]
                else:
                    dp[j] = dp[q[0]] + cost_j

            # 当前点如果可达，就加入队列
            if dp[j] < INF:
                while q and dp[q[-1]] >= dp[j]:
                    q.pop()
                q.append(j)

        out.append("Yes" if dp[-1] <= coins else "No")

    print("\n".join(out))


if __name__ == "__main__":
    main()

## C 最长回文

In [ ]:


"""
思路：
首先需要找到拼接点进行拼接:A[l1..r1] + B[l2..r2]
其中 r1=l2 整个串是回文
 
针对回文串：正着读和反着读都一样
而我们拼出来的串必须是 A 的一段 + B 的一段
从而枚举中间节点，围绕中心扩回文，找到最长回文的长度并输出
 
一个回文串可以看成:
    左边扩展部分 + 中间回文部分 + 右边扩展部分

其中:
1. 中间回文部分可能完全在 A 中
2. 中间回文部分可能完全在 B 中
3. 左右两边扩展时, 需要满足 A 往左的字符和 B 往右的字符相同


功能函数一: manacher
求每个位置作为中心时，最长回文能有多大
 
功能函数二: HashTable
O(1) 取任意子串的哈希值
 
功能函数三: extend()
求 A[a_end] 往左 和 B[b_start] 往右 最多能匹配多少个字符

做法：
1. 用 Manacher 算出 A 和 B 每个中心的最长奇/偶回文半径.
2. 用字符串哈希快速判断:
       A 从某个位置往左
       B 从某个位置往右
   最多能匹配多少字符.
3. 枚举 A 和 B 中的每个回文中心, 计算:
       中间回文长度 + 2 * 两边最大扩展长度
4. 取最大值.
"""

import sys
from array import array


def manacher(s):
    """
    返回 odd, even:
    odd[i]  = 以 i 为中心的奇回文半径, 长度为 2 * odd[i] - 1
    even[i] = 以 i 为右中心的偶回文半径, 长度为 2 * even[i]
    """
    n = len(s)

    odd = [0] * n
    even = [0] * n

    # 奇回文
    left, right = 0, -1

    for i in range(n):
        if i > right:
            radius = 1
        else:
            mirror = left + right - i
            radius = min(odd[mirror], right - i + 1)

        while i - radius >= 0 and i + radius < n and s[i - radius] == s[i + radius]:
            radius += 1

        odd[i] = radius

        if i + radius - 1 > right:
            left = i - radius + 1
            right = i + radius - 1

    # 偶回文
    left, right = 0, -1

    for i in range(n):
        if i > right:
            radius = 0
        else:
            mirror = left + right - i + 1
            radius = min(even[mirror], right - i + 1)

        while i - radius - 1 >= 0 and i + radius < n and s[i - radius - 1] == s[i + radius]:
            radius += 1

        even[i] = radius

        if i + radius - 1 > right:
            left = i - radius
            right = i + radius - 1

    return odd, even


def build_suffix_rank_levels(seq):
    """
    构建后缀数组过程中的等级表 levels。

    levels[k][i] 表示:
        从 i 开始, 长度为 2^k 的子串的排名。

    后面查 LCP 时, 可以从大到小尝试跳 2^k。
    """
    n = len(seq)

    # 初始按单个字符排序
    alphabet = max(seq) + 1
    count = [0] * alphabet

    for x in seq:
        count[x] += 1

    for i in range(1, alphabet):
        count[i] += count[i - 1]

    suffix_array = [0] * n

    for i in range(n - 1, -1, -1):
        x = seq[i]
        count[x] -= 1
        suffix_array[count[x]] = i

    rank = [0] * n
    classes = 1

    for i in range(1, n):
        if seq[suffix_array[i]] != seq[suffix_array[i - 1]]:
            classes += 1
        rank[suffix_array[i]] = classes - 1

    levels = [array("I", rank)]

    shifted = [0] * n
    new_rank = [0] * n

    k = 0

    while (1 << k) < n and classes < n:
        length = 1 << k

        # 把所有后缀位置整体向左移动 length
        for i in range(n):
            value = suffix_array[i] - length
            if value < 0:
                value += n
            shifted[i] = value

        # 按第一关键字 rank 计数排序
        count = [0] * classes

        for x in shifted:
            count[rank[x]] += 1

        position = [0] * classes
        total = 0

        for i in range(classes):
            position[i] = total
            total += count[i]

        for x in shifted:
            cls = rank[x]
            suffix_array[position[cls]] = x
            position[cls] += 1

        # 重新计算 rank
        new_rank[suffix_array[0]] = 0
        new_classes = 1

        for i in range(1, n):
            cur = suffix_array[i]
            prev = suffix_array[i - 1]

            cur_pair = (rank[cur], rank[(cur + length) % n])
            prev_pair = (rank[prev], rank[(prev + length) % n])

            if cur_pair != prev_pair:
                new_classes += 1

            new_rank[cur] = new_classes - 1

        rank, new_rank = new_rank, rank
        classes = new_classes

        levels.append(array("I", rank))

        k += 1

    return levels


def main():
    data = sys.stdin.buffer.read().split()

    n = int(data[0])
    a = data[1]
    b = data[2]

    # 1. 求 A 和 B 的所有奇偶回文半径
    odd_a, even_a = manacher(a)
    odd_b, even_b = manacher(b)

    # 2. 构造 reverse(A) + 分隔符 + B + 终止符
    # 原字符是大写字母, bytes 值加 2 后不会和 0、1 冲突
    reversed_a = a[::-1]

    seq = [x + 2 for x in reversed_a] + [1] + [x + 2 for x in b] + [0]

    levels = build_suffix_rank_levels(seq)

    total_len = len(seq)
    offset_b = n + 1

    def lcp(pos1, pos2):
        """
        求 seq[pos1:] 和 seq[pos2:] 的最长公共前缀长度。
        用后缀数组等级表从大到小跳。
        """
        if pos1 == pos2:
            return total_len - pos1

        ans = 0

        for k in range(len(levels) - 1, -1, -1):
            step = 1 << k

            if (
                pos1 + step <= total_len
                and pos2 + step <= total_len
                and levels[k][pos1] == levels[k][pos2]
            ):
                pos1 += step
                pos2 += step
                ans += step

        return ans

    def extend(a_end, b_start):
        """
        求:
            A[a_end] 往左
            B[b_start] 往右
        最多能匹配多少字符。
        """
        if a_end < 0 or b_start >= n:
            return 0

        # A[a_end] 在 reverse(A) 中的位置
        pos_in_reversed_a = n - 1 - a_end

        # B[b_start] 在 seq 中的位置
        pos_in_b = offset_b + b_start

        # 不能越过 A 左边界, 也不能越过 B 右边界
        limit = a_end + 1
        right_limit = n - b_start

        if right_limit < limit:
            limit = right_limit

        value = lcp(pos_in_reversed_a, pos_in_b)

        if value > limit:
            value = limit

        return value

    ans = 0

    # 情况一: 中间回文在 A 中, 奇回文
    for center in range(n):
        radius = odd_a[center]
        left = center - radius + 1
        right = center + radius - 1

        value = 2 * radius - 1 + 2 * extend(left - 1, right)

        if value > ans:
            ans = value

    # 情况二: 中间回文在 A 中, 偶回文
    for center in range(n):
        radius = even_a[center]
        left = center - radius
        right = center + radius - 1

        value = 2 * radius + 2 * extend(left - 1, right)

        if value > ans:
            ans = value

    # 情况三: 中间回文在 B 中, 奇回文
    for center in range(n):
        radius = odd_b[center]
        left = center - radius + 1
        right = center + radius - 1

        value = 2 * radius - 1 + 2 * extend(left, right + 1)

        if value > ans:
            ans = value

    # 情况四: 中间回文在 B 中, 偶回文
    for center in range(n):
        radius = even_b[center]
        left = center - radius
        right = center + radius - 1

        value = 2 * radius + 2 * extend(left, right + 1)

        if value > ans:
            ans = value

    print(ans)


if __name__ == "__main__":
    main()

## D 优惠券

In [ ]:
"""
思路:
这个题本质是在检查优惠券的购买和使用顺序是否合法。

正常情况下:
I x 表示买入优惠券 x, 买了之后 owned[x] = True
O x 表示使用优惠券 x, 用了之后 owned[x] = False

如果出现下面两种情况, 就会矛盾:
1. 已经有 x 了, 又出现 I x
   说明中间必须有一个坏掉的 '?' 可以当成 O x, 先把它用掉, 再买入。

2. 还没有 x, 却出现 O x
   说明中间必须有一个坏掉的 '?' 可以当成 I x, 先买入, 再使用。

所以我们需要维护:
owned[x]:
    当前是否持有优惠券 x

last_pos[x]:
    优惠券 x 上一次出现确定操作的位置

所有还没被使用的 '?':
    用树状数组维护。
    因为我们需要快速找 [last_pos[x], 当前行 i] 里面有没有可用的 '?'。

如果能找到这个 '?', 就把它消耗掉。
如果找不到, 说明当前这一行就是最早出错的位置。

为什么用树状数组:
Python 没有 C++ set.lower_bound。
树状数组可以做到:
1. 查询某个区间里有没有 '?'
2. 找到第一个 >= left 的 '?'
3. 删除这个 '?'

题目样例是多组输入, 所以这里按 EOF 一直读。
"""

import sys


class Fenwick:
    def __init__(self, n):
        self.n = n
        self.tree = [0] * (n + 1)

    def add(self, i, v):
        while i <= self.n:
            self.tree[i] += v
            i += i & -i

    def sum(self, i):
        res = 0
        while i > 0:
            res += self.tree[i]
            i -= i & -i
        return res

    def range_sum(self, l, r):
        if l > r:
            return 0
        return self.sum(r) - self.sum(l - 1)

    def kth(self, k):
        """
        找到第 k 个还没被用掉的问号的位置。
        """
        pos = 0
        bit = 1 << (self.n.bit_length() - 1)

        while bit:
            nxt = pos + bit
            if nxt <= self.n and self.tree[nxt] < k:
                pos = nxt
                k -= self.tree[nxt]
            bit >>= 1

        return pos + 1


def solve_one(records, m):
    max_x = 100005

    owned = [False] * max_x
    last_pos = [0] * max_x

    bit = Fenwick(m)

    for i in range(1, m + 1):
        op = records[i - 1][0]

        if op == b"?":
            bit.add(i, 1)
            continue

        x = int(records[i - 1][1])

        if op == b"I":
            if owned[x]:
                # 已经有 x 了, 又买 x, 中间需要一个 '?' 当成 O x
                l = last_pos[x]
                r = i

                if bit.range_sum(l, r) == 0:
                    return i

                # 找到第一个 >= l 的可用 '?'
                before = bit.sum(l - 1)
                pos = bit.kth(before + 1)
                bit.add(pos, -1)

            owned[x] = True
            last_pos[x] = i

        else:
            if not owned[x]:
                # 没有 x, 却使用 x, 中间需要一个 '?' 当成 I x
                l = last_pos[x]
                r = i

                if bit.range_sum(l, r) == 0:
                    return i

                before = bit.sum(l - 1)
                pos = bit.kth(before + 1)
                bit.add(pos, -1)

            owned[x] = False
            last_pos[x] = i

    return -1


def main():
    data = sys.stdin.buffer.read().split()
    idx = 0
    ans = []

    while idx < len(data):
        m = int(data[idx])
        idx += 1

        records = []

        for _ in range(m):
            op = data[idx]
            idx += 1

            if op == b"?":
                records.append((op,))
            else:
                x = data[idx]
                idx += 1
                records.append((op, x))

        ans.append(str(solve_one(records, m)))

    print("\n".join(ans))


if __name__ == "__main__":
    main()

## E 任意点

In [ ]:
"""
思路:
把每一个不同的 x 坐标看成一个点, 每一个不同的 y 坐标也看成一个点。

原来的一个坐标点 (x, y), 就相当于在:
    x 这个点 和 y 这个点 之间连一条边

这样整个问题就变成了一个图的问题。

比如有点:
    (1, 2)
就连边:
    X_1 -- Y_2

最后统计这个图里面有多少个连通块。
如果有 components 个连通块, 那么至少需要 components - 1 次操作才能把它们连起来。

注意:
x 和 y 的数值可能很大, 不能直接拿来当数组下标。
所以先做坐标离散化, 把出现过的 x 和 y 编号。
"""

import sys
from bisect import bisect_left


def main():
    data = sys.stdin.buffer.read().split()

    point_count = int(data[0])
    index = 1

    original_points = []
    all_x_values = []
    all_y_values = []

    for _ in range(point_count):
        x_value = int(data[index])
        y_value = int(data[index + 1])
        index += 2

        original_points.append((x_value, y_value))
        all_x_values.append(x_value)
        all_y_values.append(y_value)

    # 坐标离散化: 只保留出现过的 x 和 y
    unique_x_values = sorted(set(all_x_values))
    unique_y_values = sorted(set(all_y_values))

    x_node_count = len(unique_x_values)
    y_node_count = len(unique_y_values)

    total_node_count = x_node_count + y_node_count

    graph = [[] for _ in range(total_node_count)]

    def get_x_node_id(x_value):
        return bisect_left(unique_x_values, x_value)

    def get_y_node_id(y_value):
        return bisect_left(unique_y_values, y_value) + x_node_count

    # 每个点 (x, y) 建一条边: X_x -- Y_y
    for x_value, y_value in original_points:
        x_node_id = get_x_node_id(x_value)
        y_node_id = get_y_node_id(y_value)

        graph[x_node_id].append(y_node_id)
        graph[y_node_id].append(x_node_id)

    visited = [False] * total_node_count

    def dfs(start_node):
        stack = [start_node]
        visited[start_node] = True

        while stack:
            current_node = stack.pop()

            for next_node in graph[current_node]:
                if not visited[next_node]:
                    visited[next_node] = True
                    stack.append(next_node)

    connected_component_count = 0

    for node_id in range(total_node_count):
        if not visited[node_id]:
            connected_component_count += 1
            dfs(node_id)

    print(connected_component_count - 1)


if __name__ == "__main__":
    main()

## F 通配符匹配

In [ ]:
"""
思路:
这个题是通配符匹配。

通配符有两个:
?:
    匹配恰好一个任意字符。

*:
    匹配任意长度的字符串, 可以是空串。

因为 '*' 可以匹配任意长度, 所以先按照 '*' 把模式串切开。

例如:
    *aca?ctc

切开后就是:
    空串
    aca?ctc

又比如:
    ab?c*de??f*g

切开后就是:
    ab?c
    de??f
    g

这些片段必须按顺序出现在文件名里。

对于每个片段里的 '?':
    它可以匹配任意一个字符, 所以不用精确比较。

例如:
    ab??cd?ef

真正需要比较的只有:
    ab
    cd
    ef

所以把每个片段再拆成若干个纯字母块, 用哈希快速比较。

匹配规则:
1. 如果没有 '*':
   模式串长度必须和文件名长度一样, 然后直接匹配整个片段。

2. 如果有 '*':
   第一段必须匹配文件名前缀。
   最后一段必须匹配文件名后缀。
   中间的片段从左往右贪心找, 找到最靠左的位置就继续找下一段。

为什么中间可以贪心:
因为 '*' 可以吞掉任意长度字符。
当前片段越早匹配, 留给后面的空间越多, 不会更差。
"""

import sys

BASE = 131
MASK = (1 << 64) - 1


def build_prefix_hash(file_name):
    """
    构建文件名的前缀哈希。
    prefix_hash[i] 表示 file_name[0:i] 的哈希值。
    """
    file_length = len(file_name)
    prefix_hash = [0] * (file_length + 1)

    for i in range(file_length):
        prefix_hash[i + 1] = (prefix_hash[i] * BASE + file_name[i]) & MASK

    return prefix_hash


def get_hash(prefix_hash, power_base, left, length):
    """
    获取 file_name[left:left+length] 的哈希值。
    """
    right = left + length
    return (prefix_hash[right] - prefix_hash[left] * power_base[length]) & MASK


def build_part_chunks(pattern_part):
    """
    把一个不含 '*' 的片段按照 '?' 拆成若干个纯字母块。

    每个块保存:
        1. 块在当前片段里的起始偏移
        2. 块的长度
        3. 块的哈希值
    """
    chunks = []
    part_length = len(pattern_part)
    index = 0

    while index < part_length:
        if pattern_part[index] == ord("?"):
            index += 1
            continue

        start_index = index
        hash_value = 0

        # 连续的小写字母组成一个纯字母块
        while index < part_length and pattern_part[index] != ord("?"):
            hash_value = (hash_value * BASE + pattern_part[index]) & MASK
            index += 1

        chunks.append((start_index, index - start_index, hash_value))

    return chunks


def main():
    data = sys.stdin.buffer.read().split()

    pattern = data[0]
    file_count = int(data[1])
    file_names = data[2:2 + file_count]

    # 按 '*' 切开模式串
    pattern_parts = pattern.split(b"*")
    star_count = len(pattern_parts) - 1
    part_count = len(pattern_parts)

    # remaining_min_length[i]:
    # 从第 i 段到最后一段, 至少还需要多少字符
    # 这个用来防止中间片段匹配得太靠后
    remaining_min_length = [0] * part_count
    remaining_min_length[-1] = len(pattern_parts[-1])

    for i in range(part_count - 2, -1, -1):
        remaining_min_length[i] = remaining_min_length[i + 1] + len(pattern_parts[i])

    # 每个片段拆成纯字母块
    all_part_chunks = []

    for pattern_part in pattern_parts:
        all_part_chunks.append(build_part_chunks(pattern_part))

    # 预处理 BASE 的幂
    max_file_length = 0

    for file_name in file_names:
        if len(file_name) > max_file_length:
            max_file_length = len(file_name)

    power_base = [1] * (max_file_length + 1)

    for i in range(1, max_file_length + 1):
        power_base[i] = (power_base[i - 1] * BASE) & MASK

    answers = []

    for file_name in file_names:
        file_length = len(file_name)

        # 文件名长度必须至少能放下所有非 '*' 的部分
        if file_length < remaining_min_length[0]:
            answers.append("NO")
            continue

        prefix_hash = build_prefix_hash(file_name)

        def match_part_at_position(part_index, start_position):
            """
            判断第 part_index 个模式片段能不能从 start_position 开始匹配。
            '?' 不需要检查, 只检查纯字母块。
            """
            for chunk_offset, chunk_length, chunk_hash in all_part_chunks[part_index]:
                file_left = start_position + chunk_offset

                current_hash = get_hash(
                    prefix_hash,
                    power_base,
                    file_left,
                    chunk_length
                )

                if current_hash != chunk_hash:
                    return False

            return True

        # 没有 '*', 必须完整匹配
        if star_count == 0:
            if file_length != len(pattern_parts[0]):
                answers.append("NO")
            elif match_part_at_position(0, 0):
                answers.append("YES")
            else:
                answers.append("NO")

            continue

        # 有 '*', 先检查前缀和后缀
        first_part_length = len(pattern_parts[0])
        last_part_length = len(pattern_parts[-1])

        suffix_start_position = file_length - last_part_length

        if not match_part_at_position(0, 0):
            answers.append("NO")
            continue

        if not match_part_at_position(star_count, suffix_start_position):
            answers.append("NO")
            continue

        # 中间片段从左往右贪心匹配
        left_available_position = first_part_length
        is_valid = True

        for part_index in range(1, star_count):
            current_part_length = len(pattern_parts[part_index])

            # 当前片段最晚可以开始的位置
            max_start_position = file_length - remaining_min_length[part_index]

            found = False

            for start_position in range(left_available_position, max_start_position + 1):
                if match_part_at_position(part_index, start_position):
                    found = True
                    left_available_position = start_position + current_part_length
                    break

            if not found:
                is_valid = False
                break

        if is_valid:
            answers.append("YES")
        else:
            answers.append("NO")

    sys.stdout.write("\n".join(answers))


if __name__ == "__main__":
    main()

## G 汉诺塔

In [ ]:
"""
思路:
这个题不是选择最少步数, 而是按照题目给定的“优先级规则”去走。

规则是:
每一步都选:
1. 当前所有合法操作里优先级最高的操作
2. 不能移动上一次刚移动过的那个盘子

题目保证这样一定能把所有盘子从 A 移到 B 或 C。

关键观察:
对于只有 1 个盘子的情况, 它在某根柱子 X 上时,
下一步一定会按照优先级, 移到另外两根柱子中优先级更高的那一根。

所以先算:
preferred_target[X]:
    1 号盘在 X 柱时, 最想去哪里。

然后做 DP:

move_count[i][X]:
    有 i 个盘子都在 X 柱时,
    按这个策略一直走, 直到这 i 个盘子再次全部堆到某一根柱子上,
    需要多少步。

final_target[i][X]:
    上面这个过程结束后,
    这 i 个盘子最终会整体停在哪根柱子上。

递推时看 i 个盘子:
最大的 i 号盘在 pos_big
上面 i-1 个小盘看成一个整体, 在 pos_small

每轮过程:
1. 先让 i-1 个小盘整体按照已经算好的策略移动一次。
2. 如果小盘整体又回到了最大盘所在柱子, 说明 i 个盘子重新合成一座塔, 本轮结束。
3. 否则, 最大盘就会移动到剩下的那根空柱子。
4. 重复这个过程。

n 最大只有 30, 所以这个 DP 很小。
答案不会超过 1e18, Python 的 int 不用担心溢出。
"""

import sys


def main():
    data = sys.stdin.buffer.read().split()

    if not data:
        return

    disk_count = int(data[0])

    # 读入六种操作的优先级
    # priority[b"AB"] 越小, 表示优先级越高
    priority = {}

    data_index = 1

    for priority_rank in range(6):
        move = data[data_index]
        data_index += 1
        priority[move] = priority_rank

    # 柱子编号:
    # A -> 0
    # B -> 1
    # C -> 2
    peg_names = [b"A", b"B", b"C"]

    preferred_target = [0] * 3

    # 计算 1 号盘在每根柱子上时, 更愿意移动到哪根柱子
    for from_peg in range(3):
        first_to_peg = (from_peg + 1) % 3
        second_to_peg = (from_peg + 2) % 3

        first_move = peg_names[from_peg] + peg_names[first_to_peg]
        second_move = peg_names[from_peg] + peg_names[second_to_peg]

        if priority[first_move] < priority[second_move]:
            preferred_target[from_peg] = first_to_peg
        else:
            preferred_target[from_peg] = second_to_peg

    # move_count[i][x]:
    # i 个盘子都在 x 柱时, 走到下一次重新合成整塔需要多少步
    move_count = [[0] * 3 for _ in range(disk_count + 1)]

    # final_target[i][x]:
    # i 个盘子都从 x 柱开始, 最后整塔停在哪根柱子
    final_target = [[0] * 3 for _ in range(disk_count + 1)]

    # 只有 1 个盘子时, 一步就能移动到 preferred_target[x]
    for start_peg in range(3):
        move_count[1][start_peg] = 1
        final_target[1][start_peg] = preferred_target[start_peg]

    # 从 2 个盘子开始递推
    for current_disk_count in range(2, disk_count + 1):
        for start_peg in range(3):
            big_disk_position = start_peg
            small_tower_position = start_peg

            total_steps = 0

            while True:
                # 先移动上面的 current_disk_count - 1 个小盘子
                next_small_tower_position = final_target[current_disk_count - 1][small_tower_position]
                total_steps += move_count[current_disk_count - 1][small_tower_position]
                small_tower_position = next_small_tower_position

                # 如果小盘子又回到最大盘所在柱子, 说明重新合成一座塔
                if small_tower_position == big_disk_position:
                    final_target[current_disk_count][start_peg] = small_tower_position
                    move_count[current_disk_count][start_peg] = total_steps
                    break

                # 否则最大盘只能移动到第三根空柱子
                empty_peg = 3 - big_disk_position - small_tower_position
                big_disk_position = empty_peg
                total_steps += 1

    # 一开始所有盘子都在 A 柱, A 的编号是 0
    print(move_count[disk_count][0])


if __name__ == "__main__":
    main()

## H 马步距离

In [ ]:
"""
思路:
马走日字, 一步可以让坐标变化:
    (±1, ±2) 或 (±2, ±1)

因为棋盘无限大, 坐标也可以为负数, 所以只需要看起点和终点的距离差。
也就是说, 从 (xp, yp) 到 (xs, ys),
等价于从 (0, 0) 到 (abs(xp - xs), abs(yp - ys))。

又因为 x 和 y 可以交换, 所以令:
    x = 较大的距离
    y = 较小的距离

接下来直接用马步最短路的数学公式。

一般情况下, 最少步数至少要满足两个限制:

1. 每一步在某一个方向上最多前进 2 格,
   所以至少需要 ceil(x / 2) 步。

2. 每一步总共能让 |x| + |y| 的距离最多减少 3,
   所以至少需要 ceil((x + y) / 3) 步。

所以先取:
    ans = max(ceil(x / 2), ceil((x + y) / 3))

但是马每走一步, x + y 的奇偶性会改变。
所以最终步数 ans 的奇偶性必须和 x + y 的奇偶性一致。
如果不一致, ans 需要加 1。

还有两个特殊情况:
    (1, 0) 需要 3 步
    (2, 2) 需要 4 步

这两个点用公式会算错, 所以单独处理。
"""

import sys


def solve():
    input_data = sys.stdin.read().split()

    if not input_data:
        return

    xp, yp, xs, ys = map(int, input_data)

    # 只关心横纵坐标差
    dx = abs(xp - xs)
    dy = abs(yp - ys)

    # 利用对称性, 让 x 是较大距离, y 是较小距离
    x = max(dx, dy)
    y = min(dx, dy)

    # 特判: 从 (0, 0) 到 (1, 0) 最少需要 3 步
    if x == 1 and y == 0:
        print(3)
        return

    # 特判: 从 (0, 0) 到 (2, 2) 最少需要 4 步
    if x == 2 and y == 2:
        print(4)
        return

    # ceil(x / 2) 写成 (x + 1) // 2
    need_by_x = (x + 1) // 2

    # ceil((x + y) / 3) 写成 (x + y + 2) // 3
    need_by_total = (x + y + 2) // 3

    # 先取两个下界中的较大值
    answer = max(need_by_x, need_by_total)

    # 奇偶性修正
    # 马每走一步, x + y 的奇偶性都会变一次
    # 所以 answer 和 x + y 的奇偶性必须相同
    if answer % 2 != (x + y) % 2:
        answer += 1

    print(answer)


if __name__ == "__main__":
    solve()

## I 直方图最大矩形

In [ ]:
"""
思路:
这个题是求直方图里的最大矩形面积。

如果固定某一根柱子的高度 height 作为矩形高度,
那么这个矩形能向左、向右扩展到哪里?

答案是:
    左边第一个比它矮的柱子之后
    右边第一个比它矮的柱子之前

所以核心就是:
    对每一根柱子, 找到它左边和右边第一个比它矮的位置。

用单调递增栈来做:
    栈里存柱子的下标。
    并且栈中柱子的高度保持递增。

当遍历到当前柱子 heights[i] 时:
    如果它比栈顶柱子矮,
    说明栈顶柱子的右边界找到了,
    就可以弹出栈顶并计算面积。

为了让最后栈里剩下的柱子也能被计算,
在数组末尾加一个高度为 0 的哨兵。

时间复杂度:
    每个柱子最多入栈一次、出栈一次, 所以是 O(n)。

空间复杂度:
    栈最多存 n 个下标, 所以是 O(n)。
"""

from typing import List


class Solution:
    def largestRectangleArea(self, heights: List[int]) -> int:
        # 空数组不能形成矩形
        if not heights:
            return 0

        # 加一个哨兵 0, 强制把栈里剩下的柱子全部弹出计算
        heights = heights + [0]

        # 单调递增栈, 存的是柱子的下标
        increasing_stack = []

        max_rectangle_area = 0

        # 遍历每一根柱子
        for current_index in range(len(heights)):
            current_height = heights[current_index]

            # 当前柱子比栈顶柱子矮, 说明栈顶柱子的右边界出现了
            while increasing_stack and current_height < heights[increasing_stack[-1]]:
                rectangle_height = heights[increasing_stack.pop()]

                # 栈空, 说明左边没有更矮的柱子
                if not increasing_stack:
                    rectangle_width = current_index
                else:
                    # 新的栈顶就是左边第一个比它矮的柱子
                    left_lower_index = increasing_stack[-1]
                    rectangle_width = current_index - left_lower_index - 1

                rectangle_area = rectangle_height * rectangle_width

                if rectangle_area > max_rectangle_area:
                    max_rectangle_area = rectangle_area

            # 当前柱子入栈, 保持栈内高度递增
            increasing_stack.append(current_index)

        return max_rectangle_area

## J 消防局的设立

In [ ]:
"""
思路:
这个题是在一棵树上放消防局。

一个消防局可以覆盖距离不超过 2 的基地。
也就是说, 如果在某个点建消防局, 它可以覆盖:
    自己
    父节点 / 子节点
    爷爷节点 / 孙子节点

因为输入满足 a[i] < i, 所以父节点编号一定比子节点小。
这样我们可以从 n 到 1 倒着遍历, 相当于从叶子往根做树形 DP。

定义两个状态:

f[i]:
    在 i 的子树里, 距离 i 最近的消防局有多远。
    如果 i 的子树里没有消防局, 就是无穷大。

g[i]:
    在 i 的子树里, 距离 i 最远的还没有被覆盖的节点有多远。
    如果没有未覆盖节点, 就是负无穷。

转移过程:
1. 先合并所有子节点的信息。
2. 如果最近的消防局已经可以覆盖最远的未覆盖节点,
   也就是 f[i] + g[i] <= 2,
   那么这些未覆盖点其实已经被覆盖了。
3. 如果最远的未覆盖节点距离 i 已经等于 2,
   那么必须在 i 建消防局。
   因为如果继续往父节点传, 距离就会变成 3, 已经覆盖不到了。
4. 最后如果根节点还有未覆盖点, 就在根节点补一个消防局。

答案就是最少需要建多少个消防局。
"""

import sys


def solve():
    input_data = sys.stdin.buffer.read().split()

    if not input_data:
        return

    base_count = int(input_data[0])

    if base_count == 1:
        print(1)
        return

    # children[parent] 存 parent 的所有子节点
    children = [[] for _ in range(base_count + 1)]

    # 第 i 个输入表示 i 号节点的父节点
    for node_id in range(2, base_count + 1):
        parent_id = int(input_data[node_id - 1])
        children[parent_id].append(node_id)

    INF = 10 ** 9
    NEG_INF = -10 ** 9

    # nearest_station_distance[i]:
    # i 的子树中距离 i 最近的消防局距离
    nearest_station_distance = [INF] * (base_count + 1)

    # farthest_uncovered_distance[i]:
    # i 的子树中距离 i 最远的未覆盖节点距离
    farthest_uncovered_distance = [0] * (base_count + 1)

    station_count = 0

    # 从编号大到小遍历, 等价于自底向上处理整棵树
    for node_id in range(base_count, 0, -1):

        # 合并所有子节点的信息
        for child_id in children[node_id]:
            nearest_station_distance[node_id] = min(
                nearest_station_distance[node_id],
                nearest_station_distance[child_id] + 1
            )

            farthest_uncovered_distance[node_id] = max(
                farthest_uncovered_distance[node_id],
                farthest_uncovered_distance[child_id] + 1
            )

        # 如果当前子树里的消防局可以覆盖最远的未覆盖点
        if nearest_station_distance[node_id] + farthest_uncovered_distance[node_id] <= 2:
            farthest_uncovered_distance[node_id] = NEG_INF

        # 如果未覆盖点距离当前节点已经是 2,
        # 再往上传就会变成 3, 所以必须在当前节点建消防局
        if farthest_uncovered_distance[node_id] == 2:
            station_count += 1
            nearest_station_distance[node_id] = 0
            farthest_uncovered_distance[node_id] = NEG_INF

    # 如果根节点还有未覆盖点, 就在根节点补一个消防局
    if farthest_uncovered_distance[1] >= 0:
        station_count += 1

    print(station_count)


if __name__ == "__main__":
    solve()